# Notebook 3 — Ideal and Noisy Simulation with Qiskit Aer

**Qiskit Fall Fest 2026 — University of Ottawa**

**Difficulty:** Intermediate
**Estimated time:** 100–120 minutes
**Prerequisites:** Notebooks 1–2 (circuits, gates, measurement, observables, Sampler/Estimator).

## Learning objectives
By the end of this notebook you will be able to:

- ✓ use `AerSimulator` and Aer's V2 primitives for local simulation
- ✓ explain why real quantum hardware is noisy
- ✓ build custom noise models: bit-flip, phase-flip, depolarizing, readout, amplitude/phase damping, thermal relaxation
- ✓ visualize how increasing noise strength degrades results
- ✓ distinguish gate noise from measurement (readout) noise
- ✓ build a realistic noise model from an IBM fake backend
- ✓ compare ideal, synthetic-noise, and realistic-noise results side by side

> **Note on scope.** Everything in this notebook runs locally — no IBM Quantum account is required. Real hardware execution is covered in Notebook 4.

## 1. Why Simulators?

Even though the whole point of quantum computing is eventually running on real hardware, simulators remain essential for:

- **Learning** — build intuition without queue times or hardware costs.
- **Debugging** — verify your circuit logic against an ideal, noise-free reference before wasting hardware time on a broken circuit.
- **Ideal reference results** — you need a "ground truth" to know how much noise degraded your real results.
- **Avoiding hardware queues** — iterate quickly during a hackathon instead of waiting in line for each small change.
- **Noise studies** — deliberately study *how* noise affects specific algorithms, isolating one error source at a time.
- **Exponential classical cost** — simulators only scale to a limited number of qubits (roughly 25–30 for full statevector simulation on a laptop) since the underlying state vector doubles in size with every added qubit. This limitation is itself a key motivation for why quantum hardware matters.

## 2. AerSimulator

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Qiskit exposes this helper at different import paths across versions.
# Newer Qiskit releases support the shorter qiskit.transpiler import;
# Qiskit 1.x uses qiskit.transpiler.preset_passmanagers.
try:
    from qiskit.transpiler import generate_preset_pass_manager
except ImportError:
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

bell = QuantumCircuit(2, 2)
bell.h(0)
bell.cx(0, 1)
bell.measure([0, 1], [0, 1])

sim = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=1, backend=sim)
isa_bell = pm.run(bell)

job = sim.run(isa_bell, shots=1000)
counts = job.result().get_counts()
print(counts)

Note that we still **transpile** (`generate_preset_pass_manager(...).run(...)`) even for local ideal simulation — this is good practice, since it's the same workflow you'll use for real hardware in Notebook 4, and it ensures the circuit only uses gates the simulator/backend actually supports.

## 3. Exact vs. Shot-Based Simulation

Just as in Notebook 2: `Statevector` (or Aer's exact `EstimatorV2`) gives the **exact** mathematical answer with zero randomness. Running `sim.run(circuit, shots=n)` instead gives you **sampled** measurement outcomes, subject to statistical (shot) noise even when the simulation itself is otherwise "ideal" (noise-free).

In [ ]:
from qiskit.quantum_info import Statevector

exact_probs = Statevector(bell.remove_final_measurements(inplace=False)).probabilities_dict()
print("Exact probabilities:", exact_probs)
print("Sampled counts (1000 shots):", counts)

## 4. Aer Primitives

In [ ]:
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer.primitives import EstimatorV2 as AerEstimator

aer_sampler = AerSampler()
job = aer_sampler.run([isa_bell], shots=1000)
print("AerSampler counts:", job.result()[0].data.c.get_counts())

bell_no_meas = QuantumCircuit(2)
bell_no_meas.h(0)
bell_no_meas.cx(0, 1)
from qiskit.quantum_info import SparsePauliOp

aer_estimator = AerEstimator()
job = aer_estimator.run([(bell_no_meas, SparsePauliOp("ZZ"))])
print("AerEstimator <ZZ>:", job.result()[0].data.evs)

`qiskit_aer.primitives.SamplerV2` / `EstimatorV2` behave like the `Statevector*` primitives from Notebook 2, but run through Aer's high-performance C++ backend and — critically — **support noise models**, which the plain `Statevector*` primitives do not.

## 5. What Is Quantum Noise?

$$\text{perfect gate (mathematically exact unitary)} \quad\text{vs.}\quad \text{physical noisy operation (imperfect control pulse)}$$

Every gate on real hardware is implemented by physically manipulating a qubit (e.g. with microwave or laser pulses). This control is never perfect: pulses have finite precision, qubits interact unintentionally with their environment, and the qubits themselves slowly decay over time. The result is that a real "X gate" doesn't perfectly map $|0\rangle \to |1\rangle$ — it does so with a small, but nonzero, chance of error. Noise accumulates as circuits get longer and more qubits get involved, which is one of the central engineering challenges of building useful quantum computers today.

## 6. Qiskit Aer NoiseModel

In [ ]:
from qiskit_aer.noise import NoiseModel, pauli_error, depolarizing_error, ReadoutError, thermal_relaxation_error
print("Noise tools imported successfully.")

### Bit-Flip Error

Models $|0\rangle \leftrightarrow |1\rangle$ flips, each with probability $p$, using `pauli_error`.

In [ ]:
p_bitflip = 0.05
bitflip = pauli_error([("X", p_bitflip), ("I", 1 - p_bitflip)])

noise_bitflip = NoiseModel()
noise_bitflip.add_all_qubit_quantum_error(bitflip, ["x"])

qc0 = QuantumCircuit(1, 1)
qc0.x(0)
qc0.measure(0, 0)

sim_noisy = AerSimulator(noise_model=noise_bitflip)
pm_noisy = generate_preset_pass_manager(optimization_level=1, backend=sim_noisy)
counts = sim_noisy.run(pm_noisy.run(qc0), shots=2000).result().get_counts()
print("Bit-flip noise on X gate, 2000 shots:", counts)

> **Try it yourself.** Increase `p_bitflip` to 0.3 and re-run. How does the counts distribution change?

### Phase-Flip Error

Models a random $Z$ error. As we learned in Notebook 1, $Z$ doesn't change $Z$-basis measurement probabilities — so a naive Z-basis measurement **cannot detect this error at all**. We need to change basis (e.g. apply $H$ before measuring) to reveal it.

In [ ]:
p_phaseflip = 0.3
phaseflip = pauli_error([("Z", p_phaseflip), ("I", 1 - p_phaseflip)])

noise_phaseflip = NoiseModel()
noise_phaseflip.add_all_qubit_quantum_error(phaseflip, ["h"])

qc_z_hidden = QuantumCircuit(1, 1)
qc_z_hidden.h(0)          # prepare |+>
qc_z_hidden.measure(0, 0) # naive Z-basis measurement

sim_pf = AerSimulator(noise_model=noise_phaseflip)
pm_pf = generate_preset_pass_manager(optimization_level=1, backend=sim_pf)
counts_hidden = sim_pf.run(pm_pf.run(qc_z_hidden), shots=2000).result().get_counts()
print("Z-basis measurement (error is INVISIBLE here):", counts_hidden)

qc_z_revealed = QuantumCircuit(1, 1)
qc_z_revealed.h(0)
qc_z_revealed.h(0)       # second H reveals the phase-flip as a bit-flip
qc_z_revealed.measure(0, 0)
counts_revealed = sim_pf.run(pm_pf.run(qc_z_revealed), shots=2000).result().get_counts()
print("After a second H (error becomes VISIBLE):        ", counts_revealed)

> **Why does this matter?** This experiment is exactly why we cannot just "measure and check for errors" naively — the choice of measurement basis fundamentally determines which errors we can even detect.

### Depolarizing Error

Replaces the qubit's state with a **completely random** state with probability $p$ (and leaves it untouched with probability $1-p$) — the "worst case" generic noise model, often used as a simple stand-in for a mix of real error types.

In [ ]:
depol = depolarizing_error(0.05, 1)   # 1-qubit depolarizing error, p=0.05
noise_depol = NoiseModel()
noise_depol.add_all_qubit_quantum_error(depol, ["h"])

qc_depol = QuantumCircuit(1, 1)
qc_depol.h(0)
qc_depol.measure(0, 0)

sim_depol = AerSimulator(noise_model=noise_depol)
pm_depol = generate_preset_pass_manager(optimization_level=1, backend=sim_depol)
counts_depol = sim_depol.run(pm_depol.run(qc_depol), shots=2000).result().get_counts()
print("Depolarizing noise on H gate:", counts_depol)

### Readout Error

Models the case where the **quantum state itself is correct**, but the classical measurement apparatus reports the wrong bit — a purely classical-electronics error, distinct from any error in the quantum operations themselves.

In [ ]:
readout_err = ReadoutError([[0.9, 0.1], [0.2, 0.8]])  # P(report 0 | true 0)=0.9, P(report 1 | true 1)=0.8

noise_readout = NoiseModel()
noise_readout.add_all_qubit_readout_error(readout_err)

qc_ro = QuantumCircuit(1, 1)
qc_ro.measure(0, 0)   # qubit stays in |0>, perfect state, only readout is noisy

sim_ro = AerSimulator(noise_model=noise_readout)
pm_ro = generate_preset_pass_manager(optimization_level=1, backend=sim_ro)
counts_ro = sim_ro.run(pm_ro.run(qc_ro), shots=2000).result().get_counts()
print("Readout error on a perfect |0> state:", counts_ro)

Even though the qubit was *never* touched (it's a perfect $|0\rangle$ the whole time), about 10% of shots incorrectly report `1` — purely because of a faulty classical readout, not any quantum error.

### Amplitude Damping (Energy Relaxation)

Amplitude damping models a qubit **losing energy** and decaying from $|1\rangle$ toward $|0\rangle$ over time — this is the same physical process captured by the $T_1$ relaxation time we'll see in thermal relaxation below. Qiskit Aer implements this as a special case of `thermal_relaxation_error` (rather than a separate standalone function), so we demonstrate it via that function with $T_2 = 2T_1$ (pure amplitude damping, no extra dephasing).

In [ ]:
t1 = 50e3      # T1 in ns (arbitrary illustrative units)
t2 = 2 * t1    # T2 = 2*T1 isolates pure amplitude damping (no extra dephasing)
gate_time = 100  # ns

amp_damp = thermal_relaxation_error(t1, t2, gate_time)

noise_amp = NoiseModel()
noise_amp.add_all_qubit_quantum_error(amp_damp, ["x"])

qc_amp = QuantumCircuit(1, 1)
qc_amp.x(0)   # prepare |1>, which then decays toward |0>
qc_amp.measure(0, 0)

sim_amp = AerSimulator(noise_model=noise_amp)
pm_amp = generate_preset_pass_manager(optimization_level=1, backend=sim_amp)
counts_amp = sim_amp.run(pm_amp.run(qc_amp), shots=2000).result().get_counts()
print("Amplitude damping after preparing |1>:", counts_amp)

### Phase Damping / Dephasing

Dephasing randomizes the *phase* of a superposition over time without energy loss — this is governed by the $T_2$ time constant. We isolate pure dephasing by setting $T_1$ very large (negligible energy loss) relative to $T_2$.

In [ ]:
t1_large = 1e9   # effectively no energy relaxation
t2_short = 20e3  # much shorter T2 -> dominant dephasing

dephasing = thermal_relaxation_error(t1_large, t2_short, gate_time)

noise_deph = NoiseModel()
noise_deph.add_all_qubit_quantum_error(dephasing, ["h"])

qc_deph = QuantumCircuit(1, 1)
qc_deph.h(0)
qc_deph.h(0)  # reveals dephasing the same way it revealed the phase-flip error above
qc_deph.measure(0, 0)

sim_deph = AerSimulator(noise_model=noise_deph)
pm_deph = generate_preset_pass_manager(optimization_level=1, backend=sim_deph)
counts_deph = sim_deph.run(pm_deph.run(qc_deph), shots=2000).result().get_counts()
print("Dephasing revealed via double-H:", counts_deph)

### Thermal Relaxation

`thermal_relaxation_error(t1, t2, time)` is the general, physically realistic model combining both effects, parameterized by:

- **$T_1$** — energy relaxation time (how long until $|1\rangle$ decays toward $|0\rangle$)
- **$T_2$** — dephasing time (how long superposition phase information survives); physically, $T_2 \le 2T_1$
- **gate duration** — how long the gate takes to execute, which determines how much decay/dephasing accumulates during that gate

This is the noise model most directly tied to real physical qubit properties, and is exactly what backs the realistic fake-backend noise models in Section 10.

## 7. Noise Strength Experiment

In [ ]:
import matplotlib.pyplot as plt

def bell_circuit():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])
    return qc

probabilities = [0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.10]
correct_fraction = []

for p in probabilities:
    err = depolarizing_error(p, 2)
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(err, ["cx"])
    sim_p = AerSimulator(noise_model=nm)
    pm_p = generate_preset_pass_manager(optimization_level=1, backend=sim_p)
    counts_p = sim_p.run(pm_p.run(bell_circuit()), shots=4000).result().get_counts()
    correct = counts_p.get("00", 0) + counts_p.get("11", 0)
    correct_fraction.append(correct / 4000)

plt.figure(figsize=(6, 4))
plt.plot(probabilities, correct_fraction, marker="o")
plt.xlabel("CX depolarizing error probability")
plt.ylabel("Fraction of ideal outcomes (00 or 11)")
plt.title("Bell state degradation vs. two-qubit gate noise strength")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("noise_strength_experiment.png", dpi=150)
plt.show()

As the two-qubit gate error probability increases, the fraction of "ideal" Bell-state outcomes (`00` or `11`) visibly decreases — the noise is measurably degrading our entangled state.

## 8. Gate Noise vs. Measurement Noise

In [ ]:
nm_gate_only = NoiseModel()
nm_gate_only.add_all_qubit_quantum_error(depolarizing_error(0.05, 1), ["x"])

nm_readout_only = NoiseModel()
nm_readout_only.add_all_qubit_readout_error(ReadoutError([[0.95, 0.05], [0.05, 0.95]]))

qc_test = QuantumCircuit(1, 1)
qc_test.x(0)
qc_test.measure(0, 0)

for label, nm in [("Gate noise only", nm_gate_only), ("Readout noise only", nm_readout_only)]:
    sim_t = AerSimulator(noise_model=nm)
    pm_t = generate_preset_pass_manager(optimization_level=1, backend=sim_t)
    c = sim_t.run(pm_t.run(qc_test), shots=2000).result().get_counts()
    print(f"{label:22s}: {c}")

Both noise sources can produce similar-looking degraded counts, but they represent physically distinct mechanisms: one corrupts the *quantum operation itself*, the other corrupts only the *classical readout* after an otherwise-correct operation. Distinguishing them matters for error mitigation strategies.

## 9. One-Qubit vs. Two-Qubit Gate Errors

On real hardware, two-qubit gates (like CX) are generally implemented with **more physical steps and more susceptibility to crosstalk** than single-qubit gates, and so tend to have higher error rates in practice — though the *exact* numbers vary significantly by device and should always be checked against a specific backend's calibration data rather than assumed universally.

In [ ]:
nm_compare = NoiseModel()
nm_compare.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ["h", "x"])
nm_compare.add_all_qubit_quantum_error(depolarizing_error(0.02, 2), ["cx"])

sim_compare = AerSimulator(noise_model=nm_compare)
pm_compare = generate_preset_pass_manager(optimization_level=1, backend=sim_compare)
counts_compare = sim_compare.run(pm_compare.run(bell_circuit()), shots=4000).result().get_counts()
print("Bell state with 1q error=0.001, 2q (CX) error=0.02:", counts_compare)

## 10. Noise Model From a Fake IBM Backend

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

fake_backend = FakeSherbrooke()
fake_noise_model = NoiseModel.from_backend(fake_backend)
print(fake_noise_model)

`FakeSherbrooke` is a **fake backend** shipped with `qiskit-ibm-runtime` that mimics a real 127-qubit IBM device using a stored calibration snapshot (coupling map, basis gates, qubit $T_1$/$T_2$, gate/readout error rates). `NoiseModel.from_backend(...)` converts those calibration numbers into an Aer-compatible noise model — this gives you a **realistic** approximation of hardware behaviour without touching real hardware or needing credentials.

## 11. Compare Three Worlds: Ideal, Synthetic Noise, Fake-Backend Noise

In [ ]:
sim_ideal = AerSimulator()
sim_synthetic = AerSimulator(noise_model=nm_compare)
sim_fake = AerSimulator(noise_model=fake_noise_model)

results = {}
for label, sim_variant in [("Ideal", sim_ideal), ("Synthetic noise", sim_synthetic), ("Fake-backend noise", sim_fake)]:
    pm_variant = generate_preset_pass_manager(optimization_level=1, backend=sim_variant)
    c = sim_variant.run(pm_variant.run(bell_circuit()), shots=4000).result().get_counts()
    results[label] = c
    print(label, "->", c)

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(list(results.values()), legend=list(results.keys()))

Ideal simulation gives an almost-perfect 50/50 split between `00` and `11`. Both noise variants show visible "leakage" into the `01`/`10` outcomes that shouldn't occur ideally — but the *fake-backend* noise model gives a more physically grounded picture since it's derived from real device calibration snapshots rather than a hand-picked probability.

## 12. Transpilation and Noise

In [ ]:
ghz3 = QuantumCircuit(3)
ghz3.h(0); ghz3.cx(0, 1); ghz3.cx(1, 2)

pm_fake = generate_preset_pass_manager(optimization_level=1, backend=fake_backend)
isa_ghz3 = pm_fake.run(ghz3)

print("Logical circuit  -> depth:", ghz3.depth(), " ops:", ghz3.count_ops())
print("Transpiled (ISA) -> depth:", isa_ghz3.depth(), " ops:", isa_ghz3.count_ops())

Even though we didn't add any noise model here, notice that the **transpiled circuit is already different in structure** from the logical one — different basis gates, possibly a different depth. Since every real gate contributes some noise, a circuit's *depth* after transpilation is a rough proxy for how much noise it will accumulate. Notebook 4 covers transpilation for hardware constraints in full depth.

## 13. Exercises

**Exercise 1 — Try it yourself.** Increase the bit-flip probability in Section 6 to 0.3 and observe the counts.

**Exercise 2 — Exercise.** Build your own phase-flip noise model with a different probability and apply it to a `qc.h(0); qc.h(0); qc.measure(...)` circuit, confirming the error is revealed by the second H.

**Exercise 3 — Exercise.** Create a `ReadoutError` with different asymmetric probabilities (e.g. `[[0.99, 0.01], [0.3, 0.7]]`) and measure a perfect $|0\rangle$ and a perfect $|1\rangle$ separately.

**Exercise 4 — Exercise.** Add depolarizing noise *only* to `cx` gates (not single-qubit gates) and rerun the Bell-state experiment from Section 7.

**Exercise 5 — Exercise.** Compare 100 vs. 10,000 shots for the same noisy Bell circuit. Does the *noise level* itself change, or only the statistical precision of your estimate of it?

**Exercise 6 — Challenge.** Compare a Bell state and a 3-qubit GHZ state under the same depolarizing CX noise model. Which degrades faster, and why might that make sense given GHZ states use more CX gates?

**Exercise 7 — Challenge.** Use `FakeSherbrooke`'s noise model (Section 10) to simulate the GHZ-3 circuit and compare it against the ideal simulation using `plot_histogram`.

### Solutions (collapsed — try the exercises first!)

```python
# Exercise 1
bitflip_strong = pauli_error([("X", 0.3), ("I", 0.7)])
nm1 = NoiseModel(); nm1.add_all_qubit_quantum_error(bitflip_strong, ["x"])
sim1 = AerSimulator(noise_model=nm1)
pm1 = generate_preset_pass_manager(optimization_level=1, backend=sim1)
print(sim1.run(pm1.run(qc0), shots=2000).result().get_counts())

# Exercise 3
ro2 = ReadoutError([[0.99, 0.01], [0.3, 0.7]])
nm3 = NoiseModel(); nm3.add_all_qubit_readout_error(ro2)
sim3 = AerSimulator(noise_model=nm3)
pm3 = generate_preset_pass_manager(optimization_level=1, backend=sim3)
qc_zero = QuantumCircuit(1,1); qc_zero.measure(0,0)
qc_one = QuantumCircuit(1,1); qc_one.x(0); qc_one.measure(0,0)
print(sim3.run(pm3.run(qc_zero), shots=2000).result().get_counts())
print(sim3.run(pm3.run(qc_one), shots=2000).result().get_counts())

# Exercise 4
nm4 = NoiseModel(); nm4.add_all_qubit_quantum_error(depolarizing_error(0.05, 2), ["cx"])
sim4 = AerSimulator(noise_model=nm4)
pm4 = generate_preset_pass_manager(optimization_level=1, backend=sim4)
print(sim4.run(pm4.run(bell_circuit()), shots=4000).result().get_counts())

# Exercise 6
ghz3_meas = QuantumCircuit(3, 3); ghz3_meas.h(0); ghz3_meas.cx(0,1); ghz3_meas.cx(1,2); ghz3_meas.measure([0,1,2],[0,1,2])
sim6 = AerSimulator(noise_model=nm4)
pm6 = generate_preset_pass_manager(optimization_level=1, backend=sim6)
print(sim6.run(pm6.run(ghz3_meas), shots=4000).result().get_counts())
```

## Qiskit Cheat Sheet — Notebook 3

| Task | Code |
|---|---|
| Local ideal/noisy simulator | `AerSimulator(noise_model=nm)` |
| Aer primitives | `qiskit_aer.primitives.SamplerV2`, `EstimatorV2` |
| Noise model container | `NoiseModel()` |
| Bit/phase-flip error | `pauli_error([("X", p), ("I", 1-p)])` |
| Depolarizing error | `depolarizing_error(p, num_qubits)` |
| Readout error | `ReadoutError([[p00, p01], [p10, p11]])` |
| Thermal relaxation | `thermal_relaxation_error(t1, t2, gate_time)` |
| Attach error to gate | `nm.add_all_qubit_quantum_error(err, ["gate_name"])` |
| Attach readout error | `nm.add_all_qubit_readout_error(ro)` |
| Fake backend | `FakeSherbrooke()` |
| Noise model from backend | `NoiseModel.from_backend(fake_backend)` |
| Preset pass manager | `generate_preset_pass_manager(optimization_level, backend=sim)` |

## Common Mistakes

- **Forgetting to transpile before simulating with a noise model or fake backend.** The pass manager ensures your circuit only uses gates the backend/noise model actually defines errors for.
- **Assuming a Z-basis measurement reveals all error types.** Phase-flip and dephasing errors are invisible without a basis change (Section 6 in this notebook, Section 6 of Notebook 2).
- **Confusing gate noise with readout noise.** They are physically distinct and can be modeled/mitigated independently (Section 8).
- **Increasing shots to "reduce" noise.** More shots only reduce *statistical* uncertainty in your estimate — the underlying noisy circuit's error rate doesn't change (Exercise 5).
- **Assuming all backends have similar error rates.** Always check the fake/real backend's actual calibration data rather than assuming numbers from a different device or an old tutorial.
- **Using deprecated `qiskit.providers.aer` import paths.** Use `qiskit_aer` (separate top-level package) and `qiskit_ibm_runtime.fake_provider` for fake backends — not the older `qiskit.providers.fake_provider` or `qiskit.providers.aer` paths seen in old tutorials.